# Lesson 05: Deep Learning with PyTorch & Controlling Overfitting

In this Colab notebook, you will:
1. Use PyTorch `autograd` for automatic differentiation.
2. Build an `nn.Module` classifier with `Dropout` and `weight_decay`.
3. Complete the **Lab**: Classify MNIST digits and compare overfitting vs. regularized generalization.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Part 1: PyTorch Autograd in Action

In [ ]:
x = torch.tensor([2.0, -3.0], requires_grad=True)
y = (x ** 2).sum() + 3.0 * x[0]
y.backward()
print("Automatic gradients dy/dx:", x.grad)  # [2*2 + 3, 2*(-3)] = [7, -6]

## Part 2: Lab — Regularized MNIST MLP Classifier

In [ ]:
transform = transforms.ToTensor()
train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

class MLPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 10),
        )
    def forward(self, x):
        return self.net(x)

model = MLPClassifier().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(dim=1) == yb).sum().item()
    print(f"Epoch {epoch+1} | Test Accuracy: {100 * correct / len(test_ds):.2f}%")